In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def probabilidad_llamada(promedio_consecutivo):
    if promedio_consecutivo <= 1:
        return "Baja"
    elif 1 <= promedio_consecutivo <= 2:
        return "Media"
    else:
        return "Alta"

In [0]:
probabilidad_llamada_udf = F.udf(probabilidad_llamada, StringType())

In [0]:
df_ca_canal = spark.table(f"{catalogo}.{esquema_source}.ca_canal")
df_trx_bancarias = spark.table(f"{catalogo}.{esquema_source}.trx_bancarias")
df_trx_verificacion_llamada = spark.table(f"{catalogo}.{esquema_source}.trx_verificacion_llamada")

In [0]:
df_trx_verificacion_llamada = df_trx_verificacion_llamada.withColumn("probabilidad_llamada", probabilidad_llamada_udf("probabilidad_llamada))

In [0]:
df_joined = df_trx_bancarias.alias("x").join(df_trx_verificacion_llamada.alias("y"), col("x.nu_transaccion") == col("y.nu_transaccion"), "inner")
df_join = df_joined.alias("a").join(df_ca_canal.alias("b"), col("a.c_canal") == col("b.c_canal"), "inner")

In [0]:
df_join.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.llamadas_transaccion")